# **Cell 1 – Install & Imports**

In [0]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [0]:
%pip install catboost cloudpickle

import cloudpickle
import pandas as pd
import joblib
import os
from pyspark.sql.functions import col

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 128.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# **Cell 2 – Load All Artifacts (with Fallback)**


In [0]:
# Define volume path where artifacts are stored
VOLUME_PATH = "/Volumes/workspace/default/ml_artifacts/"

try:
    with open(VOLUME_PATH + "delivery_delay_pipeline_final.pkl", "rb") as f:
        pipeline = cloudpickle.load(f)
    logging.info("Pipeline loaded from volume")
except FileNotFoundError:
    raise FileNotFoundError(f"Missing pipeline file: {VOLUME_PATH}delivery_delay_pipeline_final.pkl")

try:
    global_avgs = joblib.load(VOLUME_PATH + "global_averages.pkl")
    logging.info("Global averages loaded from volume")
except FileNotFoundError:
    raise FileNotFoundError(f"Missing global_averages.pkl at {VOLUME_PATH}")

if os.path.exists(VOLUME_PATH + "required_features.pkl"):
    required_features = joblib.load(VOLUME_PATH + "required_features.pkl")
    logging.info(f"Required features loaded ({len(required_features)} columns) from volume")
else:
    logging.warning("required_features.pkl not found in volume – using fallback list")
    required_features = [
        'order_datetime', 'platform_name', 'product_category_name',
        'order_value_inr', 'delivery_time_min', 'service_rating',
        'sla_delay', 'refund_flag', 'Segment',
        'customer_delay_rate', 'customer_avg_rating',
        'customer_order_count', 'customer_refund_rate'
    ]

2026-05-25 13:21:43,735 - INFO - Pipeline loaded from volume
2026-05-25 13:21:43,827 - INFO - Global averages loaded from volume
2026-05-25 13:21:43,936 - INFO - Required features loaded (13 columns) from volume


# **Cell 3 – Load New Data**

In [0]:
df_orders = spark.table("silver_orders_clean").toPandas()
df_cust = spark.table("gold_customer_features").toPandas()

if df_orders.empty:
    raise ValueError("No orders found in silver_orders_clean")
if df_cust.empty:
    logging.warning("Customer table is empty – all customer features will be filled with global averages")

logging.info(f"Orders: {len(df_orders)}, Customers: {len(df_cust)}")

2026-05-25 13:21:52,575 - INFO - Orders: 100000, Customers: 9000


In [0]:
if spark.catalog.tableExists("delivery_predictions"):
    existing_orders_df = spark.table("delivery_predictions").select("order_id").distinct().toPandas()
    if not existing_orders_df.empty:
        before = len(df_orders)
        df_orders = df_orders[~df_orders['order_id'].isin(existing_orders_df['order_id'])]
        logging.info(f"Filtered out {before - len(df_orders)} already predicted orders. Remaining: {len(df_orders)}")
    else:
        logging.info("No existing orders in target table")
else:
    logging.info("Target table 'delivery_predictions' does not exist – processing all orders")

if df_orders.empty:
    logging.info("No new orders to predict. Exiting.")
    dbutils.notebook.exit("No new orders")
# ==========================================

if df_cust.empty:
    logging.warning("Customer table is empty – all customer features will be filled with global averages")

logging.info(f"Orders (new): {len(df_orders)}, Customers: {len(df_cust)}")

# **Cell 4 – Merge & Fill Nulls**

In [0]:
df = df_orders.merge(df_cust, on="customer_id", how="left")

for col, val in global_avgs.items():
    if col in df.columns:
        # Fix chained assignment
        df[col] = df[col].fillna(val)

logging.info(f"Shape after merge: {df.shape}")

2026-05-25 13:22:01,459 - INFO - Shape after merge: (0, 30)


# **Cell 5 – Keep Only Required Columns (No Manual Work)**

In [0]:
df_ids = df_orders[['order_id', 'order_datetime']].copy()   # extracted before merge to avoid duplication

existing = [c for c in required_features if c in df.columns]
missing = [c for c in required_features if c not in df.columns]

if missing:
    logging.warning(f"Missing columns: {missing}")
    for c in missing:
        if c in global_avgs:
            df[c] = global_avgs[c]
        else:
            df[c] = 0
            logging.warning(f"Column {c} not in global_avgs, filled with 0")

# Final check – all required columns must exist
assert all(c in df.columns for c in required_features), "Still missing required features after fill"

df = df[required_features].copy()
logging.info(f"Data ready: {df.shape}")

2026-05-25 13:22:05,403 - INFO - Data ready: (0, 13)


# **Cell 6 – Predict**

In [0]:
THRESHOLD = 0.5   # configurable

import time
start = time.time()

df['delay_probability'] = pipeline.predict_proba(df)[:,1]
df['predicted_delay'] = (df['delay_probability'] > THRESHOLD).astype(int)

elapsed = time.time() - start
logging.info(f"Prediction done on {len(df)} rows in {elapsed:.2f} sec")
logging.info(f"Prediction stats: mean prob = {df['delay_probability'].mean():.3f}, "
             f"p90 = {df['delay_probability'].quantile(0.9):.3f}, "
             f"positive rate = {df['predicted_delay'].mean():.3f}")

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-5713992097181515>, line 6
      3 import time
      4 start = time.time()
----> 6 df['delay_probability'] = pipeline.predict_proba(df)[:,1]
      7 df['predicted_delay'] = (df['delay_probability'] > THRESHOLD).astype(int)
      9 elapsed = time.time() - start

File /databricks/python/lib/python3.12/site-packages/sklearn/pipeline.py:903, in Pipeline.predict_proba(self, X, **params)
    901 if not _routing_enabled():
    902     for _, name, transform in self._iter(with_final=False):
--> 903         Xt = transform.transform(Xt)
    904     return self.steps[-1][1].predict_proba(Xt, **params)
    906 # metadata routing enabled

File /databricks/python/lib/python3.12/site-packages/sklearn/utils/_set_output.py:319, in _wrap_method_output.<locals>.wrapped(self, X, *args, **kwargs)
    317 @wraps(f)
    318 def wrapped(self, X, *

In [0]:
df_final = df_ids.reset_index(drop=True)
df_final['delay_probability'] = df['delay_probability'].values
df_final['predicted_delay'] = df['predicted_delay'].values

2026-04-26 06:44:54,919 - INFO - Python Server ready to receive messages
2026-04-26 06:44:54,920 - INFO - Received command c on object id p0


# **Write table (append mode + partition)**

In [0]:
spark_df = spark.createDataFrame(df_final)
spark_df.write.mode("append").partitionBy("order_datetime").saveAsTable("delivery_predictions")
logging.info("Table 'delivery_predictions' updated (append mode, partitioned by date)")

In [0]:
# Display first 10 rows of the saved table
display(spark.table("delivery_predictions"))

order_id,order_datetime,delay_probability,predicted_delay
ORD000001,2025-05-16,5.149396440166498E-4,0
ORD000004,2025-05-07,0.5799150236328915,1
ORD000013,2025-05-14,0.5649011831340579,1
ORD000015,2025-05-07,0.016835506136524746,0
ORD000055,2025-05-01,0.29036048098090017,0
ORD000075,2025-05-09,0.4871642125194818,0
ORD000082,2025-05-07,0.31721755725043566,0
ORD000091,2025-05-06,0.03476295020381069,0
ORD000122,2025-05-07,0.9221681566593483,1
ORD000129,2025-05-11,0.1264636089443463,0
